**GROUNDING DINO**

In [44]:
# 1. Scarica il file zip dalla release di GitHub
# Sostituisci il link con quello della tua release
!wget https://github.com/Lucazere00/deep_learning/releases/download/dataset/dataset.zip

# 2. Scompattalo nella directory corrente
!unzip -q dataset.zip -d /content/dataset

--2026-01-14 16:13:32--  https://github.com/Lucazere00/deep_learning/releases/download/dataset/dataset.zip
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/1117631808/dba4f1b1-853b-483f-8e2d-b159e879db91?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-01-14T17%3A06%3A02Z&rscd=attachment%3B+filename%3Ddataset.zip&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-01-14T16%3A05%3A43Z&ske=2026-01-14T17%3A06%3A02Z&sks=b&skv=2018-11-09&sig=3xM%2B%2BAcbfbM9RADLgNCGyShKWWcElJe2bd9V9vBjXV8%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc2ODQxMDgxMywibmJmIjoxNzY4NDA3MjEzLCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdGlvbi5

In [45]:
import os
HOME = os.getcwd()
print(HOME)
%cd {HOME}
!git clone https://github.com/IDEA-Research/GroundingDINO.git
!pip install pycocotools


/content/GroundingDINO
/content/GroundingDINO
Cloning into 'GroundingDINO'...
remote: Enumerating objects: 463, done.
remote: Total 463 (delta 0), reused 0 (delta 0), pack-reused 463 (from 1)
Receiving objects: 100% (463/463), 12.91 MiB | 17.96 MiB/s, done.
Resolving deltas: 100% (220/220), done.


In [46]:
%cd /content/GroundingDINO/groundingdino/models/GroundingDINO/csrc/MsDeformAttn
#!sed -i 's/value.type()/value.scalar_type()/g' ms_deform_attn_cuda.cu
#!sed -i 's/value.scalar_type().is_cuda()/value.is_cuda()/g' ms_deform_attn_cuda.cu

/content/GroundingDINO/groundingdino/models/GroundingDINO/csrc/MsDeformAttn


In [47]:
%cd {HOME}/GroundingDINO
!pip install -q -e .

/content/GroundingDINO/GroundingDINO
  Preparing metadata (setup.py) ... done


In [48]:
!mkdir weights
%cd weights
!wget -q https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth
%cd ..

/content/GroundingDINO/GroundingDINO/weights
/content/GroundingDINO/GroundingDINO


In [49]:
import cv2
from PIL import Image
import numpy as np
from torchvision.ops import box_convert
import torch
#print(torch.cuda.is_available())

from groundingdino.models import build_model
from groundingdino.util.slconfig import SLConfig
from groundingdino.util.utils import clean_state_dict
from groundingdino.util.inference import annotate, load_image, predict
import groundingdino.datasets.transforms as T

from huggingface_hub import hf_hub_download

In [50]:
def load_model_hf(repo_id, filename, ckpt_config_filename, device='cpu'):
    cache_config_file = hf_hub_download(repo_id=repo_id, filename=ckpt_config_filename)

    args = SLConfig.fromfile(cache_config_file)
    model = build_model(args)
    args.device = device

    cache_file = hf_hub_download(repo_id=repo_id, filename=filename)
    checkpoint = torch.load(cache_file, map_location='cpu')
    log = model.load_state_dict(clean_state_dict(checkpoint['model']), strict=False)
    print("Model loaded from {} \n => {}".format(cache_file, log))
    _ = model.eval()
    return model

# Grounding DINO model
ckpt_repo_id = "ShilongLiu/GroundingDINO"
ckpt_filenmae = "groundingdino_swint_ogc.pth"
ckpt_config_filename = "GroundingDINO_SwinT_OGC.cfg.py"


In [51]:
model = load_model_hf(ckpt_repo_id, ckpt_filenmae, ckpt_config_filename)

final text_encoder_type: bert-base-uncased
Model loaded from /root/.cache/huggingface/hub/models--ShilongLiu--GroundingDINO/snapshots/a94c9b567a2a374598f05c584e96798a170c56fb/groundingdino_swint_ogc.pth 
 => _IncompatibleKeys(missing_keys=[], unexpected_keys=['label_enc.weight', 'bert.embeddings.position_ids'])


In [52]:
import os
import shutil

# -----------------------------
# PATH DI OUTPUT
# -----------------------------
OUTPUT_DIR = "/content/dataset/dataset/predictions"
ANNOTATED_DIR = os.path.join(OUTPUT_DIR, "annotated_images")
OUTPUT_JSON = os.path.join(OUTPUT_DIR, "grounding_dino_test_predictions.json")

# -----------------------------
# RESET OUTPUT (elimina tutto e ricrea le cartelle)
# -----------------------------
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(ANNOTATED_DIR, exist_ok=True)

print("Cartelle di output resettate e pronte.")


Cartelle di output resettate e pronte.


In [53]:
import os
import json
import time
from tqdm import tqdm
import cv2
import torch
import random

# -----------------------------
# PARAMETRI
# -----------------------------
TEXT_PROMPT = "person . weapon"
BOX_THRESHOLD = 0.30
TEXT_THRESHOLD = 0.25
DEVICE = "cpu"  # oppure "cuda"

IMAGE_DIR = "/content/dataset/dataset/test"
OUTPUT_DIR = "/content/dataset/dataset/predictions"
ANNOTATED_DIR = os.path.join(OUTPUT_DIR, "annotated_images")
OUTPUT_JSON = os.path.join(OUTPUT_DIR, "grounding_dino_test_predictions.json")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(ANNOTATED_DIR, exist_ok=True)


# -----------------------------
# FUNZIONE: cxcywh -> xyxy pixel
# -----------------------------
def cxcywh_to_xyxy(boxes, W, H):
    boxes_xyxy = []
    for box in boxes:
        cx, cy, w, h = box
        x1 = (cx - w / 2) * W
        y1 = (cy - h / 2) * H
        x2 = (cx + w / 2) * W
        y2 = (cy + h / 2) * H
        boxes_xyxy.append([x1.item(), y1.item(), x2.item(), y2.item()])
    return boxes_xyxy

# -----------------------------
# LOOP DI INFERENZA
# -----------------------------
predictions = []
times = []

image_files = sorted(os.listdir(IMAGE_DIR))


for img_file in tqdm(image_files, desc="Inferenza Grounding DINO"):
    img_path = os.path.join(IMAGE_DIR, img_file)

    try:
        image_source, image = load_image(img_path)
    except:
        print(f"Immagine non caricabile: {img_file}")
        continue

    H, W = image_source.shape[:2]

    start = time.time()
    boxes, logits, phrases = predict(
        model=model,
        image=image,
        caption=TEXT_PROMPT,
        box_threshold=BOX_THRESHOLD,
        text_threshold=TEXT_THRESHOLD,
        device=DEVICE
    )
    infer_time = time.time() - start
    times.append(infer_time)

    # Salvataggio JSON
    if len(boxes) == 0:
        predictions.append({
            "image_name": img_file,
            "boxes": [],
            "scores": [],
            "labels": []
        })
    else:
        boxes_xyxy = cxcywh_to_xyxy(boxes, W, H)
        predictions.append({
            "image_name": img_file,
            "boxes": boxes_xyxy,
            "scores": logits.cpu().tolist(),
            "labels": phrases
        })

        # -----------------------------
        # Salvataggio immagine annotata
        # -----------------------------
        annotated_frame = annotate(
            image_source=image_source,
            boxes=boxes,
            logits=logits,
            phrases=phrases
        )
        annotated_frame = annotated_frame[..., ::-1]  # BGR -> RGB
        save_path = os.path.join(ANNOTATED_DIR, img_file)
        cv2.imwrite(save_path, annotated_frame)

    # Log per immagine
    print(f"{img_file} | box: {len(boxes)} | time: {infer_time:.3f}s")

# -----------------------------
# SALVATAGGIO JSON COMPLETO
# -----------------------------
with open(OUTPUT_JSON, "w") as f:
    json.dump(predictions, f, indent=2)

print("\n--- INFERENZA COMPLETATA ---")
print(f"Immagini processate: {len(predictions)}")
print(f"Tempo medio: {sum(times)/len(times):.3f} s")
print(f"File JSON salvato in: {OUTPUT_JSON}")
print(f"Immagini annotate salvate in: {ANNOTATED_DIR}")


Inferenza Grounding DINO:   0%|          | 1/500 [00:34<4:47:48, 34.61s/it]

OSOpvoUTO6A_30_60_000004_jpg.rf.a60a0c875e5d47e85caddd20db5a0ad4.jpg | box: 2 | time: 34.496s


Inferenza Grounding DINO:   0%|          | 2/500 [00:58<3:54:01, 28.20s/it]

OSOpvoUTO6A_30_60_000021_jpg.rf.503fb9a5ec616755b797d1ab44272895.jpg | box: 1 | time: 23.676s


Inferenza Grounding DINO:   1%|          | 3/500 [01:22<3:39:05, 26.45s/it]

...

sc81bdNyLRA_30_60_000028_jpg.rf.f5ceae720e879497842388d54bef4c33.jpg | box: 3 | time: 23.187s


Inferenza Grounding DINO: 100%|█████████▉| 499/500 [3:16:53<00:23, 23.63s/it]

sc81bdNyLRA_30_60_000056_jpg.rf.38775ec573a68ce6cc545cb97e3a0f9c.jpg | box: 3 | time: 23.297s


Inferenza Grounding DINO: 100%|██████████| 500/500 [3:17:17<00:00, 23.67s/it]

sc81bdNyLRA_30_60_000062_jpg.rf.1369eacbcaa69ba4b5f69b7fc7d16234.jpg | box: 2 | time: 23.604s

--- INFERENZA COMPLETATA ---
Immagini processate: 500
Tempo medio: 23.638 s
File JSON salvato in: /content/dataset/dataset/predictions/grounding_dino_test_predictions.json
Immagini annotate salvate in: /content/dataset/dataset/predictions/annotated_images


In [54]:
!pip install pycocotools

In [58]:
import json
from pycocotools.coco import COCO

# -----------------------------
# PATH
# -----------------------------
GT_JSON = "/content/dataset/dataset/annotations/instances_test2017.json"
PRED_JSON = "/content/dataset/dataset/predictions/grounding_dino_test_predictions.json"
OUTPUT_COCO_PRED = "/content/dataset/dataset/predictions/grounding_dino_coco_predictions.json"

# -----------------------------
# LOAD COCO GT
# -----------------------------
coco_gt = COCO(GT_JSON)

# mappa filename -> image_id
file_to_image_id = {img["file_name"]: img["id"] for img in coco_gt.dataset["images"]}

# -----------------------------
# LABEL MAPPING
# -----------------------------
LABEL_TO_COCO_ID = {
    "person": 1,
    "weapon": 2
}
# -----------------------------
# LOAD PREDICTIONS
# -----------------------------
with open(PRED_JSON, "r") as f:
    preds = json.load(f)

coco_predictions = []

for p in preds:
    file_name = p["image_name"]
    if file_name not in file_to_image_id:
        continue
    image_id = file_to_image_id[file_name]

    for box, score, label in zip(p["boxes"], p["scores"], p["labels"]):
        if label not in LABEL_TO_COCO_ID:
            continue
        x1, y1, x2, y2 = box
        w = x2 - x1
        h = y2 - y1
        coco_predictions.append({
            "image_id": image_id,
            "category_id": LABEL_TO_COCO_ID[label],
            "bbox": [x1, y1, w, h],
            "score": score
        })

# -----------------------------
# SAVE COCO PREDICTIONS
# -----------------------------
with open(OUTPUT_COCO_PRED, "w") as f:
    json.dump(coco_predictions, f)

print("Predizioni COCO salvate:", OUTPUT_COCO_PRED)


loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Predizioni COCO salvate: /content/dataset/dataset/predictions/grounding_dino_coco_predictions.json


In [59]:
from pycocotools.cocoeval import COCOeval

coco_dt = coco_gt.loadRes(OUTPUT_COCO_PRED)

coco_eval = COCOeval(coco_gt, coco_dt, iouType="bbox")
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()

Loading and preparing results...
DONE (t=0.02s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.37s).
Accumulating evaluation results...
DONE (t=0.13s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.681
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.761
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.704
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.354
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.291
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.793
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.637
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.720
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.720
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=10

In [60]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# -----------------------------
# PATH
# -----------------------------
GT_JSON = "/content/dataset/dataset/annotations/instances_test2017.json"
PRED_JSON = "/content/dataset/dataset/predictions/grounding_dino_coco_predictions.json"  # assicurati formato COCO

# -----------------------------
# CATEGORIE
# -----------------------------
CLASSES = {
    1: "person",
    2: "weapon"
}

# -----------------------------
# CARICA COCO
# -----------------------------
coco_gt = COCO(GT_JSON)
coco_dt = coco_gt.loadRes(PRED_JSON)

# -----------------------------
# FUNZIONE PER CALCOLARE P/R PER CLASSE
# -----------------------------
def evaluate_class(cat_id):
    coco_eval = COCOeval(coco_gt, coco_dt, iouType='bbox')
    coco_eval.params.catIds = [cat_id]
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()
    stats = coco_eval.stats
    result = {
        "class": CLASSES[cat_id],
        "AP IoU=0.50:0.95": stats[0],
        "AP IoU=0.50": stats[1],
        "AP IoU=0.75": stats[2],
        "AR maxDets=100": stats[8]  # Recall media
    }
    return result

# -----------------------------
# VALUTAZIONE TUTTE LE CLASSI
# -----------------------------
results = []
for cid in CLASSES.keys():
    print(f"\n--- Valutazione classe: {CLASSES[cid]} ---")
    res = evaluate_class(cid)
    results.append(res)

# -----------------------------
# STAMPA RISULTATI IN TABELLA
# -----------------------------
print("\n--- Tabella Precision / Recall per classe ---")
for r in results:
    print(f"{r['class']:>6} | AP@[0.5:0.95]={r['AP IoU=0.50:0.95']:.3f} | AP@0.5={r['AP IoU=0.50']:.3f} | "
          f"AP@0.75={r['AP IoU=0.75']:.3f} | AR (Recall)={r['AR maxDets=100']:.3f}")


loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!

--- Valutazione classe: person ---
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.15s).
Accumulating evaluation results...
DONE (t=0.05s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.880
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.943
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.898
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.700
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.324
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.894
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.836
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.906
 Average Recall     (AR) @[ I